## Version 4: Market Readiness
### Objective
Version 2 scored countries on electricity need and economic capacity. Version 4 asks a separate question:
**can a private mini-grid developer actually operate in the country?**

Readiness is kept outside the opportunity score, so a large electricity gap cannot hide a missing
legal, licensing or tariff framework.

### Market-readiness design

The model distinguishes between:

1. **Minimum feasibility conditions**, which determine whether a country can be
   assessed as currently investable for mini-grid development; and

2. **Readiness-strength factors**, which differentiate feasible markets based on
   the quality of their enabling environment.

High electricity need alone does not imply near-term investment feasibility if
the regulatory pathway, tariff framework, or protection against grid arrival is unclear.

### Evidence rules for market readiness

Each criterion will be assessed using a consistent, verifiable source and recorded
as `Yes`, `No`, or `Unclear`.

| Criterion | Operational definition | RISE question | Proposed treatment |
|---|---|---|---|
| Private ownership / operation | Private companies may own and/or operate mini-grids. | 3.2.6 | Feasibility gate |
| Mini-grid / off-grid regulation | A law, regulation, or official framework explicitly governs mini-grid or off-grid electricity development. | 3.3.1 | Feasibility gate |
| Licensing pathway | An official process specifies whether and how a developer obtains a licence, permit, registration, or exemption. | 3.3.2 | Feasibility gate |
| Tariff framework | Developers may charge tariffs under a defined approval, cap, negotiated, or cost-reflective process. | 3.3.4 | Feasibility gate |
| Grid-arrival provisions | Official rules address what happens when the main grid reaches a mini-grid service area, such as compensation, asset transfer, or interconnection. | 3.3.3 | Readiness-strength factor |
| National DRE support | A current national electrification, energy, or mini-grid plan explicitly recognises decentralised renewable energy or mini-grids. | 1.3.1 | Readiness-strength factor |

A country will be classified as **currently feasible** only where all four feasibility gates are evidenced. A missing source is recorded as `Unclear`, not assumed
to mean `No`.

### Candidate cross-country source: RISE

The Regulatory Indicators for Sustainable Energy (RISE), developed by the World Bank
and ESMAP, will be evaluated as the primary cross-country source for market-readiness
evidence.

RISE is suitable for initial screening because it provides comparable country-level
indicators on sustainable-energy policy and regulation, including a mini-grid topic.

The assessment will test whether the available RISE mini-grid data can consistently
support the following criteria across the 48-country SSA sample:

- mini-grid / off-grid regulation;
- licensing pathway;
- tariff framework;
- grid-arrival provisions; and
- national support for decentralised renewable energy.

RISE results will be used for initial classification only. Where a country is
shortlisted, its current official regulations and electrification plans must be
checked directly, because regulatory frameworks can change after the RISE assessment
year.

In [1]:
# Import libraries and point Python to the project folder,
# so functions in src/ can be imported
import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd().parent
sys.path.append(str(project_root))

### 1. Rebuild the Version 2 dataset
The pipeline rebuilds the 48-country SSA dataset (electricity access, population and
GDP per capita) directly from the raw World Bank files.

In [2]:
# Build the Version 2 dataset: 48 SSA countries, including GDP_Score
# Expected shape: (48, 87); most columns are the raw 1960-2025 year columns
from src.pipeline import prepare_investment_dataset

ssa_df = prepare_investment_dataset()
ssa_df.shape

(48, 87)

### 2. Load the RISE mini-grid data
Source: World Bank/ESMAP RISE, Electricity Access pillar, sheet `EA 2023`.
The file contains overall scores, sub-indicator scores and the Yes/No answer to every
RISE question for 57 countries, with one sheet per year (2010–2023).

The first RISE download (`rise-data-score-minigrid.xls`) was replaced because it only
contained one overall score per country.

In [3]:
# Import the RISE loader from src/load_data.py
from src.load_data import load_rise_data

In [4]:
# Load the 2023 sheet ('Scoring System' row is removed inside load_rise_data)
# Expected shape: (57, 84)
rise_raw = load_rise_data(
    project_root / "data" / "raw" / "market readiness" / "rise electricity report.xlsx"
)
rise_raw.shape

(57, 84)

### 3. Select and rename the RISE columns
Only 11 of the 84 columns are needed: the mini-grid scores, the four feasibility gates,
two readiness factors and viability gap funding.

Columns are matched by their RISE question number (e.g. `3.3.4`) instead of their full
text, because the question texts are long and some contain trailing spaces.

In [5]:
# RISE question number -> short column name
# len(rename_map) must be 11; anything less means a question number did not match
rise_codes = {
    "3": "RISE_Minigrid_Score",
    "3.1": "RISE_Strategy_Score",
    "3.2": "RISE_Financing_Score",
    "3.3": "RISE_Regulations_Score",
    "3.2.6": "Gate_Private_Operation",
    "3.3.1": "Gate_Legal",
    "3.3.2": "Gate_Licensing",
    "3.3.4": "Gate_Tariff",
    "3.3.3": "Grid_Arrival_Rules",
    "1.3.1": "Plan_Includes_DRE",
    "3.2.1": "Viability_Gap_Funding",
}

rename_map = {
    column: new_name
    for column in rise_raw.columns
    for code, new_name in rise_codes.items()
    if column.startswith(f"{code}: ")
}

len(rename_map)

11

In [6]:
# Keep the 11 RISE columns plus the country name, then shorten the column names
# Fix the two country names that are spelt differently from the World Bank data
# Expected shape: (57, 12)
rise_name_fixes = {
    "Côte d'Ivoire": "Cote d'Ivoire",
    "Somalia": "Somalia, Fed. Rep.",
}

rise_df = (
    rise_raw[["Country"] + list(rename_map)]
    .rename(columns=rename_map)
    .rename(columns={"Country": "Country Name"})
)

rise_df["Country Name"] = rise_df["Country Name"].replace(rise_name_fixes)

rise_df.shape

(57, 12)

### 4. Match RISE countries to the SSA dataset
Only two SSA names differed between the sources (Côte d'Ivoire and Somalia); they are
fixed above with an explicit mapping rather than fuzzy matching.

RISE covers **35 of the 48** SSA countries. The other 13 are not in the RISE Electricity
Access pillar: Botswana, Cabo Verde, Comoros, Equatorial Guinea, Eswatini, Gabon, Gambia,
Guinea-Bissau, Lesotho, Mauritius, Namibia, São Tomé and Príncipe, Seychelles.

In [7]:
# How many RISE countries match the SSA list? Expected: 35
rise_df["Country Name"].isin(ssa_df["Country Name"]).sum()

np.int64(35)

### 5. Merge RISE into the SSA dataset
All 48 SSA countries are kept. Countries without RISE data are labelled
**"Not assessed by RISE"** instead of being left blank, because missing data is not
evidence of a weak framework.

**Why RISE is not added to the weighted opportunity score:**
1. Coverage is 35/48 (~73%), below the 90% rule set in Version 3.
2. The missing countries are not random (mostly small or high-access economies).
   Filling them with 0 or an average would invent data; dropping them would break
   comparability with Versions 1 and 2.
3. A weighted sum lets high electricity need offset a missing legal or tariff framework
   (e.g. South Sudan scores 5/100 on RISE). Readiness works as a gate, not as a trade-off.
4. Keeping need and readiness separate gives a clearer two-dimensional view.

Trade-off: Version 4 no longer produces a single top-10 list.

In [8]:
# Left merge: keep all 48 SSA countries, attach RISE columns where available
# Then label each country as assessed / not assessed by RISE
# Expected shape: (48, 99) = 87 columns + 11 RISE columns + RISE_Status
readiness_df = ssa_df.merge(rise_df, on="Country Name", how="left")

readiness_df["RISE_Status"] = np.where(
    readiness_df["RISE_Minigrid_Score"].notna(),
    "Assessed",
    "Not assessed by RISE",
)

readiness_df.shape

(48, 99)

In [9]:
# Check: 35 Assessed, 13 Not assessed by RISE
readiness_df["RISE_Status"].value_counts()

RISE_Status
Assessed                35
Not assessed by RISE    13
Name: count, dtype: int64

### 6. Classify mini-grid feasibility

| Category | Rule |
|---|---|
| Currently feasible | All four gates are Yes: private operation, legal status, licensing, cost-reflective tariff |
| Feasible with subsidy support | Private operation, legal status and licensing are Yes; tariff is No; publicly funded viability gap funding exists (RISE 3.2.1) |
| Fails a gate | Any other assessed country with at least one gate answered No |
| Not assessed | Country not covered by RISE |

The subsidy category exists because a capped tariff does not stop mini-grids from
operating when the revenue gap is publicly funded.

In [10]:
# Four feasibility gates (all must be Yes to be 'Currently feasible')
gate_columns = [
    "Gate_Private_Operation",
    "Gate_Legal",
    "Gate_Licensing",
    "Gate_Tariff",
]

# True/False per country: covered by RISE? passes every gate?
assessed = readiness_df["RISE_Status"].eq("Assessed")
passes_all_gates = readiness_df[gate_columns].eq("Yes").all(axis=1)

# Subsidy route: core gates pass, tariff fails, viability gap funding exists
core_gates = ["Gate_Private_Operation", "Gate_Legal", "Gate_Licensing"]

subsidy_route = (
    readiness_df[core_gates].eq("Yes").all(axis=1)
    & readiness_df["Gate_Tariff"].eq("No")
    & readiness_df["Viability_Gap_Funding"].eq("Yes")
)

# np.select checks conditions in order and uses the first one that is True
# Expected: 19 currently feasible, 4 with subsidy support, 12 fail a gate, 13 not assessed
readiness_df["Feasibility"] = np.select(
    [~assessed, passes_all_gates, subsidy_route],
    ["Not assessed", "Currently feasible", "Feasible with subsidy support"],
    default="Fails a gate",
)

readiness_df["Feasibility"].value_counts()

Feasibility
Currently feasible               19
Not assessed                     13
Fails a gate                     12
Feasible with subsidy support     4
Name: count, dtype: int64

### 7. Which gates do countries fail?
A "Fails a gate" label alone does not say why, so the failed gates are listed by name.

In [11]:
# List the gates each country answered 'No' to (e.g. 'Tariff', 'Licensing')
# Countries that pass, or are not assessed, get an empty string
readiness_df["Failed_Gates"] = (
    readiness_df[gate_columns]
    .eq("No")
    .apply(
        lambda row: ", ".join(
            column.replace("Gate_", "") for column in row.index[row]
        ),
        axis=1,
    )
)

# Show the countries that fail, highest RISE score first
readiness_df.loc[
    readiness_df["Feasibility"] == "Fails a gate",
    ["Country Name", "RISE_Minigrid_Score", "Failed_Gates"],
].sort_values("RISE_Minigrid_Score", ascending=False)
readiness_df["Failed_Gates"] = readiness_df["Failed_Gates"].replace("", "None")

**Finding: tariff is the binding constraint.**
Before the subsidy category was added, 16 assessed countries failed at least one gate:
6 on tariff alone, 5 on licensing alone, and only South Sudan on several (legal, licensing, tariff).
Every country scoring above 60 on RISE that failed a gate failed on tariff, which the
overall RISE score hides (e.g. Angola and Senegal both score 83).

Four of the tariff-only countries have viability gap funding (Senegal, Sierra Leone,
Mozambique, Togo) and move to "Feasible with subsidy support". The remaining 12 are
shown above.

### 8. Final opportunity score
The opportunity score combines three indicators with equal weights:
access deficit, people without electricity, and GDP per capita.
Improvement is excluded (see section 9).

**Scope rule:** countries with 2023 electricity access above 90% are out of scope for a
mini-grid screen (Seychelles, Mauritius, Cabo Verde, Gabon). Scores are rescaled within the
44 in-scope countries, so out-of-scope countries do not set the scale for the rest.
None of these four is assessed by RISE either, which is consistent with the cut-off.

Every blank score carries a reason in `Score_Status` instead of an unexplained NaN.

In [12]:
# Final opportunity score: access, population, GDP (one third each; improvement is context only)
# Scored only for countries with 2023 access at or below 90%; scores are rescaled within that group
from src.scoring import (
    calculate_access_score,
    calculate_population_score,
    calculate_gdp_score,
    calculate_weighted_score,
)

in_scope = readiness_df["2023"] <= 90

scope_df = readiness_df[in_scope].copy()
scope_df = calculate_access_score(scope_df)
scope_df = calculate_population_score(scope_df)
scope_df = calculate_gdp_score(scope_df)
scope_df = calculate_weighted_score(
    scope_df,
    {"Access_Score": 1/3, "Population_Score": 1/3, "GDP_Score": 1/3},
    output_column="Opportunity_Score",
)
scope_df["Opportunity_Rank"] = scope_df["Opportunity_Score"].rank(ascending=False, method="min")

# Attach only the two new columns to the full 48-country table
readiness_df = readiness_df.merge(
    scope_df[["Country Code", "Opportunity_Score", "Opportunity_Rank"]],
    on="Country Code",
    how="left",
)

# Explain every blank score instead of leaving NaN unexplained
readiness_df["Score_Status"] = np.select(
    [~in_scope, readiness_df["Opportunity_Score"].isna()],
    ["Out of scope (access above 90%)", "Not scored (missing GDP data)"],
    default="Scored",
)

readiness_df["Score_Status"].value_counts()

Score_Status
Scored                             42
Out of scope (access above 90%)     4
Not scored (missing GDP data)       2
Name: count, dtype: int64

In [13]:
readiness_df.sort_values("Opportunity_Rank")[["Country Name", "Opportunity_Score", "Opportunity_Rank", "Feasibility"]].head(15)

,Country Name,Opportunity_Score,Opportunity_Rank,Feasibility
8,"Congo, Dem. Rep.",0.672097,1.0,Currently feasible
31,Nigeria,0.654077,2.0,Currently feasible
41,Chad,0.500489,3.0,Currently feasible
13,Ethiopia,0.491666,4.0,Currently feasible
3,Burkina Faso,0.457902,5.0,Fails a gate
0,Angola,0.457231,6.0,Fails a gate
28,Malawi,0.450079,7.0,Currently feasible
43,Tanzania,0.447429,8.0,Currently feasible
30,Niger,0.446182,9.0,Currently feasible
19,Equatorial Guinea,0.412139,10.0,Not assessed


**Result:** DRC, Nigeria, Ethiopia, Malawi, Tanzania and Niger combine high opportunity
with current feasibility. Angola and Burkina Faso rank highly on opportunity but fail the
tariff gate without viability gap funding. Equatorial Guinea, Namibia and Botswana rank
on economic capacity alone and have no RISE readiness data.

**Limitation:** the model does not yet include political, security or macroeconomic risk.
Chad ranks 3rd on opportunity and passes all regulatory gates, but its operating risk
is not captured. Risk is the next candidate dimension.

### 9. Decision record: removing improvement from the opportunity score
Version 3 demoted access improvement (2014–2023) to a contextual indicator, because its
direction is ambiguous. The cells below compare the opportunity score with improvement
(25% each, as in Version 2) and without it (one third each for access, population, GDP).

**Result:** without improvement, slow improvers rise (Burkina Faso +18, Chad +8) and fast
improvers fall (Rwanda −33, Kenya −26). Kenya's fall comes entirely from losing its
improvement score, as most of its access gap is already closed. Kenya's strength is
captured by RISE instead (score 100, passes all gates).

**Decision:** improvement is removed from the score and kept as context.

**Open issue:** without improvement, GDP's weight rises to one third, which lifts
high-income, high-access countries (Equatorial Guinea, Seychelles). Whether near-universal-access
countries belong in a mini-grid screen at all is still to be decided.

In [14]:
# Calculate the opportunity score twice: with improvement (Version 2 weights)
# and without improvement (equal thirds for access, population, GDP)
from src.scoring import calculate_investment_opportunity_score

readiness_df = calculate_investment_opportunity_score(
    readiness_df,
    access_weight=0.25,
    improvement_weight=0.25,
    population_weight=0.25,
    gdp_weight=0.25,
    output_column="Score_With_Improvement",
)

readiness_df = calculate_investment_opportunity_score(
    readiness_df,
    access_weight=1/3,
    improvement_weight=0,
    population_weight=1/3,
    gdp_weight=1/3,
    output_column="Score_Without_Improvement",
)

# Rank both scores; positive Rank_Change = country moves up when improvement is dropped
comparison = readiness_df[
    ["Country Name", "Score_With_Improvement", "Score_Without_Improvement"]
].copy()

comparison["Rank_With"] = comparison["Score_With_Improvement"].rank(ascending=False, method="min")
comparison["Rank_Without"] = comparison["Score_Without_Improvement"].rank(ascending=False, method="min")
comparison["Rank_Change"] = comparison["Rank_With"] - comparison["Rank_Without"]

comparison.sort_values("Rank_Without").head(15)

,Country Name,Score_With_Improvement,Score_Without_Improvement,Rank_With,Rank_Without,Rank_Change
8,"Congo, Dem. Rep.",0.549237,0.660783,1.0,1.0,0.0
31,Nigeria,0.522629,0.637104,3.0,2.0,1.0
13,Ethiopia,0.526311,0.485672,2.0,3.0,-1.0
41,Chad,0.384563,0.478090,12.0,4.0,8.0
3,Burkina Faso,0.350835,0.441231,23.0,5.0,18.0
28,Malawi,0.354229,0.436907,19.0,6.0,13.0
43,Tanzania,0.469989,0.435649,4.0,7.0,-3.0
30,Niger,0.356284,0.434485,18.0,8.0,10.0
0,Angola,0.434530,0.430406,8.0,9.0,-1.0
25,Mozambique,0.351277,0.377662,22.0,10.0,12.0


In [15]:
# The 8 biggest movers, up or down (sorted by the size of the rank change)
comparison.reindex(comparison["Rank_Change"].abs().sort_values(ascending=False).index).head(8)

,Country Name,Score_With_Improvement,Score_Without_Improvement,Rank_With,Rank_Without,Rank_Change
32,Rwanda,0.444037,0.258716,7.0,40.0,-33.0
22,Lesotho,0.361042,0.255725,15.0,42.0,-27.0
20,Kenya,0.448243,0.293085,6.0,32.0,-26.0
19,Equatorial Guinea,0.292945,0.375844,34.0,12.0,22.0
3,Burkina Faso,0.350835,0.441231,23.0,5.0,18.0
2,Benin,0.363859,0.308154,14.0,30.0,-16.0
47,Zimbabwe,0.415145,0.326388,9.0,25.0,-16.0
40,Seychelles,0.256084,0.333333,40.0,24.0,16.0


In [16]:
# Explain the movers: compare two risers (Burkina Faso, Chad) with two fallers (Kenya, Rwanda)
readiness_df.loc[
    readiness_df["Country Name"].isin(["Burkina Faso", "Chad", "Kenya", "Rwanda"]),
    ["Country Name", "2014", "2023", "Improvement", "Access_Score", "Population_Score", "GDP_Score"],
]

,Country Name,2014,2023,Improvement,Access_Score,Population_Score,GDP_Score
3,Burkina Faso,19.2,21.7,2.5,0.827696,0.203907,0.292090
20,Kenya,36.0,76.2,40.2,0.251586,0.148958,0.478710
32,Rwanda,19.8,63.9,44.1,0.381607,0.056974,0.337568
41,Chad,8.4,12.0,3.6,0.930233,0.192276,0.311762


In [17]:
# All 48 countries by 2023 access rate, highest first,
# with people without electricity to show how much market is left
readiness_df[["Country Name", "2023", "People_Without_Electricity", "Feasibility"]].sort_values(
    "2023", ascending=False
).head(15)

,Country Name,2023,People_Without_Electricity,Feasibility
40,Seychelles,100.0,0.000000e+00,Not assessed
27,Mauritius,100.0,0.000000e+00,Not assessed
11,Cabo Verde,98.6,7.312634e+03,Not assessed
14,Gabon,94.1,1.466026e+05,Not assessed
10,Comoros,89.8,8.673947e+04,Not assessed
15,Ghana,89.5,3.547731e+06,Currently feasible
45,South Africa,87.7,7.775123e+06,Fails a gate
39,Eswatini,86.4,1.673488e+05,Not assessed
38,Sao Tome and Principe,81.3,4.317288e+04,Not assessed
20,Kenya,76.2,1.317068e+07,Currently feasible


### 10. Export the Version 4 dataset
One row per SSA country (48), trimmed to the columns used in the dashboard.
The intermediate 48-country component scores are deliberately excluded: the final
`Opportunity_Score` is scaled within the 44 in-scope countries, and mixing the two
scales would be misleading.

In [18]:
# Columns for Power BI: identifiers, access trend, model inputs,
# final score with status, RISE readiness, and feasibility outcome
from src.clean_data import select_years

export_columns = (
    ["Country Name", "Country Code", "IncomeGroup"]
    + select_years(2014, 2023)
    + [
        "Population_2023",
        "People_Without_Electricity",
        "Improvement",
        "GDP_Per_Capita_2023",
        "Opportunity_Score",
        "Opportunity_Rank",
        "Score_Status",
        "RISE_Status",
        "RISE_Minigrid_Score",
        "RISE_Strategy_Score",
        "RISE_Financing_Score",
        "RISE_Regulations_Score",
        "Gate_Private_Operation",
        "Gate_Legal",
        "Gate_Licensing",
        "Gate_Tariff",
        "Viability_Gap_Funding",
        "Grid_Arrival_Rules",
        "Plan_Includes_DRE",
        "Feasibility",
        "Failed_Gates",
    ]
)

v4_export = readiness_df[export_columns].copy()

# Save to data/processed, next to the Version 1 and 2 outputs
# Expected shape: (48, 34)
output_path = project_root / "data" / "processed" / "ssa_market_readiness_v4.csv"
v4_export.to_csv(output_path, index=False)

v4_export.shape

(48, 34)